# TFM — Agente conversacional ReAct para atención bancaria en Ecuador

**Autora:** Johanna García Mora  
**Director:** Joaquín Gaspar Medina Arco  
**Institución:** UNIR — Máster Universitario en Inteligencia Artificial  
**Versión:** v2.0

---

## Descripción del experimento

Evaluación comparativa de un agente ReAct (LangGraph + Groq) frente a un sistema de referencia basado en reglas (baseline_v2).  
- **18 escenarios** Banking77-EC  
- **3 repeticiones** por escenario y sistema  
- **108 ejecuciones totales**  
- **Métrica:** cobertura léxica con lematización (spaCy es_core_news_sm)


In [1]:
# Celda 1: Instalación de librerías
!pip install langchain langgraph langchain-groq spacy -q
!python -m spacy download es_core_news_sm -q
print('Librerías instaladas correctamente')

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Librerías instaladas correctamente


In [2]:
# Celda 2: Configuración de API Key
import os
from google.colab import userdata

# La clave se lee desde los Secrets de Colab (nunca hardcodeada)
# En Colab: panel izquierdo → icono llave → añadir GROQ_API_KEY
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('API Key configurada correctamente')

API Key configurada correctamente


In [3]:
# Celda 3: Conexión con Groq y prueba del modelo
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0,
    api_key=os.environ['GROQ_API_KEY']
)

respuesta = llm.invoke('Hola, ¿puedes responder en una frase breve?')
print('Modelo conectado:', respuesta.content[:80])

Modelo conectado: Sí, puedo responder en una frase breve.


In [4]:
# Celda 4: Agente ReAct con herramientas bancarias
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def consultar_politica_bancaria(categoria: str) -> str:
    """Consulta la política bancaria oficial para una categoría de servicio."""
    politicas = {
        'bloqueo': 'Política de bloqueo: el cliente puede bloquear su tarjeta llamando al 1800-BANCO o desde la app. Se genera número de caso automáticamente. La reposición tarda 5 días hábiles.',
        'credito': 'Política de crédito: requisitos incluyen cédula, rol de pagos de los últimos 3 meses y planilla de servicios. Tasa desde 15.2% anual. Capacidad máxima de endeudamiento: 40% de ingresos netos.',
        'actualizacion': 'Política de actualización de datos: cambios de contacto se realizan por app con OTP. Cambios de empresa requieren documentación en sucursal. Plazo: 3-5 días hábiles.',
        'certificados': 'Política de certificados: certificado de cuenta activa cuesta $2 y se emite en 24h por app o sucursal. Certificado de no adeudar requiere verificación previa.',
        'claves': 'Política de claves: recuperación de clave web mediante OTP al email registrado. PIN bloqueado se puede resetear por app, cajero o sucursal.',
        'desgravamen': 'Política de desgravamen: el seguro se cancela automáticamente al pagar el crédito. Si fue cobrado sin autorización, el banco reembolsa hasta 36 meses anteriores.'
    }
    categoria_lower = categoria.lower()
    for key, val in politicas.items():
        if key in categoria_lower:
            return val
    return 'No se encontró política específica para esta categoría. Derivar a ejecutivo de cuenta.'

@tool
def verificar_identidad(cedula: str) -> str:
    """Verifica la identidad del cliente mediante su número de cédula."""
    if len(cedula) == 10 and cedula.isdigit():
        return f'Identidad verificada para cédula {cedula}. Cliente activo en el sistema.'
    return 'Cédula inválida. Solicitar documento físico en sucursal.'

@tool
def generar_numero_caso(tipo: str) -> str:
    """Genera un número de caso para seguimiento de la solicitud del cliente."""
    import random
    numero = f'CASO-2026-{random.randint(10000,99999)}'
    return f'Caso generado: {numero} para {tipo}. El cliente puede hacer seguimiento con este número.'

herramientas = [consultar_politica_bancaria, verificar_identidad, generar_numero_caso]
agente = create_react_agent(llm, herramientas)
print('Agente ReAct creado con 3 herramientas bancarias')

Agente ReAct creado con 3 herramientas bancarias


/tmp/ipykernel_20828/1645292569.py:37: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(llm, herramientas)


In [5]:
# Celda 5: Baseline_v2 — sistema de referencia corregido
# Correcciones respecto a baseline_v1:
# 1. Límites de palabra (\b) para evitar que 'desbloquear' active regla de 'bloquear'
# 2. Regla independiente para 'desgravamen' con prioridad sobre 'crédito'
# 3. Regla independiente para 'fraude' con prioridad sobre 'bloqueo'
# 4. Lematización con spaCy para 'bloqueo'/'bloquear', 'crédito'/'crediticio'
# 5. Sinónimos: tarjeta/débito/crédito para bloqueo; pin/contraseña/clave para claves

import re
import spacy

nlp = spacy.load('es_core_news_sm')

def lematizar(texto: str) -> str:
    """Lematiza el texto usando spaCy."""
    doc = nlp(texto.lower())
    return ' '.join([token.lemma_ for token in doc])

def sistema_baseline_v2(consulta: str) -> dict:
    """
    Baseline_v2: sistema de referencia corregido.
    Mejoras: límites de palabra, lematización, sinónimos, reglas separadas.
    """
    consulta_lower = consulta.lower()
    consulta_lema = lematizar(consulta)

    # PRIORIDAD 1: Fraude (antes de bloqueo genérico)
    if re.search(r'\b(fraude|robo|robaron|hurtaron|no autoriz)\b', consulta_lower):
        return {
            'respuesta': 'Por seguridad bloqueamos su tarjeta inmediatamente. Se genera caso de reclamación por fraude. El área de seguridad se contactará en 24h. Puede hacer seguimiento con su número de caso.',
            'categoria': 'fraude',
            'resuelto': True,
            'pasos': 1
        }

    # PRIORIDAD 2: Desgravamen (antes de crédito genérico)
    if re.search(r'\b(desgravamen|seguro de vida|seguro del crédito)\b', consulta_lower):
        return {
            'respuesta': 'El seguro de desgravamen se cancela automáticamente al liquidar el crédito. Si fue cobrado sin autorización, puede reclamar reembolso de hasta 36 meses. Acérquese a sucursal con su cédula.',
            'categoria': 'desgravamen',
            'resuelto': True,
            'pasos': 1
        }

    # PRIORIDAD 3: Desbloqueo (\b evita confusión con bloqueo)
    if re.search(r'\b(desbloquear|desbloqueo|desbloqué)\b', consulta_lower) or \
       re.search(r'\b(desbloquear|desbloqueo|desbloqué)\b', consulta_lema):
        return {
            'respuesta': 'Para desbloquear su tarjeta puede hacerlo desde la app bancaria o llamando al 1800-BANCO. También puede acudir a cualquier sucursal con su cédula y tarjeta.',
            'categoria': 'desbloqueo',
            'resuelto': True,
            'pasos': 1
        }

    # PRIORIDAD 4: Bloqueo (\b evita match en 'desbloquear')
    if re.search(r'\b(bloquear|bloqueo|bloqueé|bloqué|bloquear|perdí|robaron la tarjeta)\b', consulta_lower) or \
       re.search(r'\b(bloquear|bloqueo)\b', consulta_lema):
        return {
            'respuesta': 'Procedemos a bloquear su tarjeta de inmediato. Llame al 1800-BANCO o use la app. Se generará un número de caso para seguimiento y la reposición llega en 5 días hábiles.',
            'categoria': 'bloqueo',
            'resuelto': True,
            'pasos': 1
        }

    # PRIORIDAD 5: Crédito
    if re.search(r'\b(crédito|préstamo|crediticio|financiamiento|hipotecario|consumo)\b', consulta_lower) or \
       re.search(r'\b(crédito|préstamo)\b', consulta_lema):
        return {
            'respuesta': 'Para créditos necesita: cédula, rol de pagos últimos 3 meses y planilla de servicios. La tasa es desde 15.2% anual. La capacidad máxima de endeudamiento es el 40% de sus ingresos netos.',
            'categoria': 'credito',
            'resuelto': True,
            'pasos': 1
        }

    # PRIORIDAD 6: Actualización de datos
    if re.search(r'\b(actualizar|cambiar|modificar|actualización)\b', consulta_lower) and \
       re.search(r'\b(datos|teléfono|dirección|empresa|información)\b', consulta_lower):
        return {
            'respuesta': 'Para actualizar datos de contacto use la app con OTP. Para cambios de empresa acuda a sucursal con documentación. El plazo es de 3 a 5 días hábiles.',
            'categoria': 'actualizacion',
            'resuelto': True,
            'pasos': 1
        }

    # PRIORIDAD 7: Certificados
    if re.search(r'\b(certificado|constancia|documento bancario)\b', consulta_lower):
        return {
            'respuesta': 'El certificado de cuenta activa cuesta $2 y se emite en 24h por app o sucursal. Para certificado de no adeudar se realiza verificación previa en 1 día hábil.',
            'categoria': 'certificados',
            'resuelto': True,
            'pasos': 1
        }

    # PRIORIDAD 8: Claves y PIN
    if re.search(r'\b(clave|pin|contraseña|password|acceso)\b', consulta_lower):
        return {
            'respuesta': 'Para recuperar su clave web use el OTP enviado a su email. El PIN bloqueado puede resetearse por app, cajero o sucursal. Para sistemas internos contacte a mesa de ayuda.',
            'categoria': 'claves',
            'resuelto': True,
            'pasos': 1
        }

    # Sin coincidencia
    return {
        'respuesta': 'No puedo resolver esta consulta con la información disponible. Por favor contacte a un ejecutivo de cuenta.',
        'categoria': 'no_identificado',
        'resuelto': False,
        'pasos': 1
    }

print('Baseline_v2 definido con límites de palabra, lematización y reglas separadas')
# Prueba rápida de corrección de bugs
r = sistema_baseline_v2('Quiero desbloquear mi tarjeta')
print(f'Test desbloqueo: categoria={r["categoria"]} (esperado: desbloqueo)')
r2 = sistema_baseline_v2('Tengo un seguro de desgravamen no solicitado')
print(f'Test desgravamen: categoria={r2["categoria"]} (esperado: desgravamen)')

Baseline_v2 definido con límites de palabra, lematización y reglas separadas
Test desbloqueo: categoria=desbloqueo (esperado: desbloqueo)
Test desgravamen: categoria=desgravamen (esperado: desgravamen)


In [6]:
# Celda 6: Métrica — cobertura léxica con lematización
import spacy
nlp = spacy.load('es_core_news_sm')

def cobertura_lexica(respuesta: str, elementos_esperados: list) -> float:
    """
    Calcula la cobertura léxica: proporción de elementos esperados
    presentes en la respuesta, con lematización.
    Nota: métrica de cobertura de forma léxica, no evaluación semántica real.
    """
    respuesta_lema = lematizar(respuesta)
    encontrados = 0
    for elem in elementos_esperados:
        elem_lema = lematizar(elem)
        # Verificar forma original o lematizada
        if elem.lower() in respuesta.lower() or elem_lema in respuesta_lema:
            encontrados += 1
    return round(encontrados / len(elementos_esperados), 2)

print('Función cobertura_lexica definida con lematización spaCy')

Función cobertura_lexica definida con lematización spaCy


In [7]:
# Celda 7: Definición de los 18 escenarios Banking77-EC
escenarios = [
    # CATEGORÍA 1: Bloqueo/Desbloqueo
    {'id': 'E1.1', 'categoria': 'Bloqueo tarjeta', 'complejidad': 'Baja',
     'consulta': 'Necesito bloquear mi tarjeta de débito, la perdí esta mañana en el transporte. Mi cédula es 1723456789.',
     'elementos_esperados': ['bloqueo', 'caso', 'reposición'],
     'politica_disponible': True},
    {'id': 'E1.2', 'categoria': 'Desbloqueo tarjeta', 'complejidad': 'Media',
     'consulta': 'Quiero desbloquear mi tarjeta de crédito, la bloqueé por error desde la app. Cédula 0912345678.',
     'elementos_esperados': ['desbloqueo', 'verificar', 'pin'],
     'politica_disponible': True},
    {'id': 'E1.3', 'categoria': 'Bloqueo por fraude', 'complejidad': 'Alta',
     'consulta': 'Mi tarjeta fue robada ayer y hicieron compras no autorizadas por $847. Cédula 1756789012.',
     'elementos_esperados': ['bloqueo', 'fraude', 'reclamación', 'caso'],
     'politica_disponible': True},

    # CATEGORÍA 2: Crédito
    {'id': 'E2.1', 'categoria': 'Requisitos crédito', 'complejidad': 'Baja',
     'consulta': '¿Cuáles son los requisitos para solicitar un crédito de consumo de $5.000?',
     'elementos_esperados': ['cédula', 'requisitos', 'tasa'],
     'politica_disponible': True},
    {'id': 'E2.2', 'categoria': 'Estado solicitud', 'complejidad': 'Media',
     'consulta': '¿En qué estado está mi solicitud de crédito hipotecario #SOL-2026-1547?',
     'elementos_esperados': ['estado', 'etapa', 'tiempo'],
     'politica_disponible': False},
    {'id': 'E2.3', 'categoria': 'Capacidad endeudamiento', 'complejidad': 'Alta',
     'consulta': 'Tengo ingresos de $2.000, crédito vigente de $8.000 con cuota de $198. ¿Puedo acceder a $3.000 adicionales?',
     'elementos_esperados': ['capacidad', '40%', 'cuota', 'califica'],
     'politica_disponible': True},

    # CATEGORÍA 3: Actualización datos
    {'id': 'E3.1', 'categoria': 'Actualizar contacto', 'complejidad': 'Baja',
     'consulta': 'Necesito actualizar mi número de teléfono y dirección en el sistema del banco.',
     'elementos_esperados': ['OTP', 'app', 'sucursal'],
     'politica_disponible': True},
    {'id': 'E3.2', 'categoria': 'Actualizar empresa', 'complejidad': 'Media',
     'consulta': 'Quiero actualizar los datos de mi empresa porque cambié de razón social.',
     'elementos_esperados': ['RUC', 'documentos', 'sucursal', 'días'],
     'politica_disponible': True},
    {'id': 'E3.3', 'categoria': 'Cuenta bloqueada', 'complejidad': 'Alta',
     'consulta': 'Mi empleado no puede actualizar sus datos bancarios porque tiene la cuenta bloqueada por inactividad.',
     'elementos_esperados': ['inactividad', 'reactivación', 'RRHH', 'plazo'],
     'politica_disponible': False},

    # CATEGORÍA 4: Certificados
    {'id': 'E4.1', 'categoria': 'Certificado cuenta', 'complejidad': 'Baja',
     'consulta': 'Necesito un certificado de cuenta activa para presentar al consulado.',
     'elementos_esperados': ['costo', 'emisión', 'app', 'sucursal'],
     'politica_disponible': True},
    {'id': 'E4.2', 'categoria': 'Certificado no adeudar', 'complejidad': 'Media',
     'consulta': 'Necesito un certificado de que no tengo deudas vencidas para un concurso público.',
     'elementos_esperados': ['vencidas', 'verificación', 'costo', 'tiempo'],
     'politica_disponible': True},
    {'id': 'E4.3', 'categoria': 'Certificado retenciones', 'complejidad': 'Alta',
     'consulta': 'Soy del área contable y necesito certificados de retenciones 2025 para 5 empleados. El SRI vence en 10 días.',
     'elementos_esperados': ['portal', 'masiva', 'días hábiles', 'SRI'],
     'politica_disponible': False},

    # CATEGORÍA 5: Claves
    {'id': 'E5.1', 'categoria': 'Reseteo clave web', 'complejidad': 'Baja',
     'consulta': 'Olvidé mi clave de la banca web, ¿cómo la recupero?',
     'elementos_esperados': ['OTP', 'email', 'nueva clave'],
     'politica_disponible': True},
    {'id': 'E5.2', 'categoria': 'PIN bloqueado', 'complejidad': 'Media',
     'consulta': 'Mi PIN de cajero está bloqueado después de tres intentos fallidos. ¿Qué hago?',
     'elementos_esperados': ['app', 'cajero', 'sucursal', 'NFC'],
     'politica_disponible': True},
    {'id': 'E5.3', 'categoria': 'Clave sistema interno', 'complejidad': 'Alta',
     'consulta': 'Soy jefe de agencia y necesito resetear la clave del sistema interno de un operador que lleva 2 horas sin poder trabajar.',
     'elementos_esperados': ['mesa de ayuda', 'ticket', 'autoridad', 'urgente'],
     'politica_disponible': False},

    # CATEGORÍA 6: Desgravamen
    {'id': 'E6.1', 'categoria': 'Cancelar desgravamen', 'complejidad': 'Media',
     'consulta': '¿Cómo cancelo el seguro de desgravamen de mi crédito que ya pagué en su totalidad?',
     'elementos_esperados': ['automática', 'reembolso', 'primas'],
     'politica_disponible': True},
    {'id': 'E6.2', 'categoria': 'Seguro no solicitado', 'complejidad': 'Alta',
     'consulta': 'Me están cobrando $18.50 mensual por un seguro de desgravamen que nunca pedí, desde hace 8 meses.',
     'elementos_esperados': ['reclamación', 'reembolso', '148', 'legal'],
     'politica_disponible': True},
    {'id': 'E6.3', 'categoria': 'Siniestro fallecimiento', 'complejidad': 'Alta',
     'consulta': 'Un cliente falleció y tiene crédito vigente con saldo $4.200 y seguro de desgravamen activo. ¿Cómo aplicamos el seguro?',
     'elementos_esperados': ['defunción', 'documentos', 'días hábiles', 'intereses'],
     'politica_disponible': True},
]

print(f'Banco de escenarios cargado: {len(escenarios)} escenarios Banking77-EC')

Banco de escenarios cargado: 18 escenarios Banking77-EC


In [8]:
# Celda 8: EXPERIMENTO COMPLETO — 3 repeticiones × 18 escenarios × 2 sistemas = 108 ejecuciones
import time
import statistics

N_REPETICIONES = 3
resultados_completos = []  # Una entrada por (escenario, repetición)

print('=' * 65)
print('EXPERIMENTO COMPARATIVO — AGENTE ReAct vs BASELINE_V2')
print(f'Escenarios: {len(escenarios)} | Repeticiones: {N_REPETICIONES} | Total: {len(escenarios)*N_REPETICIONES*2} ejecuciones')
print('=' * 65)

for escenario in escenarios:
    print(f'\n📋 {escenario["id"]} — {escenario["categoria"]} ({escenario["complejidad"]})')
    coberturas_baseline = []
    coberturas_react = []

    for rep in range(1, N_REPETICIONES + 1):
        # --- BASELINE_V2 ---
        t0 = time.perf_counter()
        res_b = sistema_baseline_v2(escenario['consulta'])
        lat_b = round(time.perf_counter() - t0, 4)
        cob_b = cobertura_lexica(res_b['respuesta'], escenario['elementos_esperados'])
        lon_b = len(res_b['respuesta'].split())
        coberturas_baseline.append(cob_b)

        # --- AGENTE ReAct ---
        t0 = time.perf_counter()
        try:
            res_r = agente.invoke({
                'messages': [
                    {'role': 'system', 'content': 'Eres un agente de soporte bancario. Resuelve la consulta usando las herramientas disponibles. Responde en español.'},
                    {'role': 'user', 'content': escenario['consulta']}
                ]
            })
            resp_r = res_r['messages'][-1].content
            pasos_r = sum(1 for m in res_r['messages'] if 'ToolMessage' in type(m).__name__)
            exito_r = True
        except Exception as e:
            resp_r = f'Error: {str(e)}'
            pasos_r = 0
            exito_r = False
        lat_r = round(time.perf_counter() - t0, 2)
        cob_r = cobertura_lexica(resp_r, escenario['elementos_esperados'])
        lon_r = len(resp_r.split())
        coberturas_react.append(cob_r)

        resultados_completos.append({
            'id': escenario['id'],
            'categoria': escenario['categoria'],
            'complejidad': escenario['complejidad'],
            'politica_disponible': escenario['politica_disponible'],
            'repeticion': rep,
            'baseline': {'cobertura': cob_b, 'latencia': lat_b, 'longitud': lon_b, 'pasos': 1},
            'react': {'cobertura': cob_r, 'latencia': lat_r, 'longitud': lon_r, 'pasos': pasos_r, 'exito': exito_r}
        })

        print(f'   Rep {rep}: Baseline={cob_b:.0%} ({lat_b}s) | ReAct={cob_r:.0%} ({lat_r}s, {pasos_r} herram.)')

    # Resumen del escenario
    print(f'   MEDIA: Baseline={statistics.mean(coberturas_baseline):.0%} | ReAct={statistics.mean(coberturas_react):.0%} | SD_ReAct={statistics.stdev(coberturas_react) if len(coberturas_react)>1 else 0:.2f}')

print(f'\n✅ EXPERIMENTO COMPLETADO — {len(resultados_completos)} ejecuciones registradas')

EXPERIMENTO COMPARATIVO — AGENTE ReAct vs BASELINE_V2
Escenarios: 18 | Repeticiones: 3 | Total: 108 ejecuciones

📋 E1.1 — Bloqueo tarjeta (Baja)
   Rep 1: Baseline=67% (0.0238s) | ReAct=100% (1.32s, 3 herram.)
   Rep 2: Baseline=67% (0.0097s) | ReAct=33% (1.19s, 3 herram.)
   Rep 3: Baseline=67% (0.0069s) | ReAct=67% (1.18s, 3 herram.)
   MEDIA: Baseline=67% | ReAct=67% | SD_ReAct=0.34

📋 E1.2 — Desbloqueo tarjeta (Media)
   Rep 1: Baseline=0% (0.0077s) | ReAct=33% (1.58s, 3 herram.)
   Rep 2: Baseline=0% (0.0054s) | ReAct=67% (1.3s, 3 herram.)
   Rep 3: Baseline=0% (0.0056s) | ReAct=67% (4.67s, 2 herram.)
   MEDIA: Baseline=0% | ReAct=56% | SD_ReAct=0.20

📋 E1.3 — Bloqueo por fraude (Alta)
   Rep 1: Baseline=0% (0.0084s) | ReAct=25% (14.35s, 3 herram.)
   Rep 2: Baseline=0% (0.0058s) | ReAct=25% (13.62s, 3 herram.)
   Rep 3: Baseline=0% (0.0057s) | ReAct=25% (14.86s, 3 herram.)
   MEDIA: Baseline=0% | ReAct=25% | SD_ReAct=0.00

📋 E2.1 — Requisitos crédito (Baja)
   Rep 1: Baseline=67%

In [9]:
# Celda 9: Análisis de resultados agregados
import statistics
from scipy import stats

# Cobertura media por escenario (promedio de 3 repeticiones)
ids = list(dict.fromkeys(r['id'] for r in resultados_completos))
cob_b_por_escenario = []
cob_r_por_escenario = []

for eid in ids:
    filas = [r for r in resultados_completos if r['id'] == eid]
    cob_b_por_escenario.append(statistics.mean(r['baseline']['cobertura'] for r in filas))
    cob_r_por_escenario.append(statistics.mean(r['react']['cobertura'] for r in filas))

print('=' * 60)
print('RESULTADOS AGREGADOS (media de 3 repeticiones por escenario)')
print('=' * 60)
print(f'\nBaseline_v2 — Cobertura léxica media: {statistics.mean(cob_b_por_escenario):.1%}')
print(f'Agente ReAct — Cobertura léxica media: {statistics.mean(cob_r_por_escenario):.1%}')

# Wilcoxon
stat, p = stats.wilcoxon(cob_r_por_escenario, cob_b_por_escenario)
n = len(ids)
r_effect = stat / (n * (n + 1) / 2)
print(f'\nTest de Wilcoxon: W={stat:.1f}, p={p:.3f}, r={r_effect:.2f}')
print(f'Interpretación: {"diferencia significativa (p<0.05)" if p < 0.05 else "sin diferencia significativa"}')
print(f'Tamaño del efecto r={r_effect:.2f}: {"pequeño" if r_effect<0.3 else "mediano" if r_effect<0.5 else "grande"}')

# Por complejidad
print('\nPor nivel de complejidad:')
for comp in ['Baja', 'Media', 'Alta']:
    filas = [r for r in resultados_completos if r['complejidad'] == comp]
    ids_comp = list(dict.fromkeys(r['id'] for r in filas))
    cb = [statistics.mean(r['baseline']['cobertura'] for r in filas if r['id']==eid) for eid in ids_comp]
    cr = [statistics.mean(r['react']['cobertura'] for r in filas if r['id']==eid) for eid in ids_comp]
    print(f'  {comp}: Baseline={statistics.mean(cb):.1%} | ReAct={statistics.mean(cr):.1%}')

# Por disponibilidad de política
print('\nPor disponibilidad de política:')
for disp, label in [(True, 'Con política'), (False, 'Sin política')]:
    filas = [r for r in resultados_completos if r['politica_disponible'] == disp]
    ids_disp = list(dict.fromkeys(r['id'] for r in filas))
    cb = [statistics.mean(r['baseline']['cobertura'] for r in filas if r['id']==eid) for eid in ids_disp]
    cr = [statistics.mean(r['react']['cobertura'] for r in filas if r['id']==eid) for eid in ids_disp]
    print(f'  {label}: Baseline={statistics.mean(cb):.1%} | ReAct={statistics.mean(cr):.1%}')

# Latencia y longitud ReAct
lats = [r['react']['latencia'] for r in resultados_completos]
lons_r = [r['react']['longitud'] for r in resultados_completos]
lons_b = [r['baseline']['longitud'] for r in resultados_completos]
pasos = [r['react']['pasos'] for r in resultados_completos]
print(f'\nLatencia ReAct: media={statistics.mean(lats):.1f}s, máx={max(lats):.1f}s')
print(f'Longitud media: ReAct={statistics.mean(lons_r):.0f} tokens | Baseline={statistics.mean(lons_b):.0f} tokens')
print(f'Herramientas por ejecución: media={statistics.mean(pasos):.1f}')

print('\n✅ Análisis completado')

RESULTADOS AGREGADOS (media de 3 repeticiones por escenario)

Baseline_v2 — Cobertura léxica media: 38.5%
Agente ReAct — Cobertura léxica media: 26.7%

Test de Wilcoxon: W=22.0, p=0.181, r=0.13
Interpretación: sin diferencia significativa
Tamaño del efecto r=0.13: pequeño

Por nivel de complejidad:
  Baja: Baseline=70.2% | ReAct=35.6%
  Media: Baseline=36.2% | ReAct=36.6%
  Alta: Baseline=17.9% | ReAct=11.9%

Por disponibilidad de política:
  Con política: Baseline=45.9% | ReAct=30.4%
  Sin política: Baseline=12.5% | ReAct=13.8%

Latencia ReAct: media=7.4s, máx=18.3s
Longitud media: ReAct=112 tokens | Baseline=28 tokens
Herramientas por ejecución: media=1.4

✅ Análisis completado


In [10]:
# Celda 10: Detalle por escenario
print(f'{"ID":<6} {"Categoría":<25} {"Comp":<6} {"Baseline":>9} {"ReAct":>9} {"Δ":>7}')
print('-' * 65)
for eid, cb, cr in zip(ids, cob_b_por_escenario, cob_r_por_escenario):
    escenario = next(e for e in escenarios if e['id'] == eid)
    delta = cr - cb
    signo = '+' if delta >= 0 else ''
    print(f'{eid:<6} {escenario["categoria"]:<25} {escenario["complejidad"]:<6} {cb:>8.0%} {cr:>8.0%} {signo}{delta:>6.0%}')

ID     Categoría                 Comp    Baseline     ReAct       Δ
-----------------------------------------------------------------
E1.1   Bloqueo tarjeta           Baja        67%      67%    -0%
E1.2   Desbloqueo tarjeta        Media        0%      56% +   56%
E1.3   Bloqueo por fraude        Alta         0%      25% +   25%
E2.1   Requisitos crédito        Baja        67%      67% +    0%
E2.2   Estado solicitud          Media        0%      22% +   22%
E2.3   Capacidad endeudamiento   Alta        50%       0%   -50%
E3.1   Actualizar contacto       Baja       100%       0%  -100%
E3.2   Actualizar empresa        Media       50%       0%   -50%
E3.3   Cuenta bloqueada          Alta        25%      25% +    0%
E4.1   Certificado cuenta        Baja        50%      33%   -17%
E4.2   Certificado no adeudar    Media       25%      25% +    0%
E4.3   Certificado retenciones   Alta         0%       8% +    8%
E5.1   Reseteo clave web         Baja        67%      11%   -56%
E5.2   PIN blo